# 23 · Optional grouped binary-support prediction sets

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Secondary adapted analysis, not a reproduction of a named paper or a selective-error guarantee. Reserve its calibration protocol before final testing.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Load a fixed model’s independent group-calibration claims

In [ ]:
from oncoplate.calibration import group_prediction_set_threshold,binary_support_sets
import numpy as np
calibration=read_table(p['private']/'conformal_support_calibration.csv')
assert calibration.split.eq('calibration').all()
assert calibration.support_status.isin(['supported','contradicted','unverifiable']).all()
y=calibration.support_status.eq('supported').to_numpy(int)
threshold=group_prediction_set_threshold(calibration.p_supported.to_numpy(float),y,calibration.group_id.to_numpy(),alpha=.10)
write_json(p['private']/'group_prediction_set_threshold.json',threshold)
print(threshold)

## 2. Inspect set-valued predictions on development cases

In [ ]:
validation=read_table(p['private']/'conformal_support_validation.csv')
assert not set(validation.group_id)&set(calibration.group_id)
sets=binary_support_sets(validation.p_supported.to_numpy(float),threshold)
validation['includes_unsupported']=sets[:,0];validation['includes_supported']=sets[:,1]
write_table(p['reports']/'optional_group_prediction_sets_validation.csv',validation)
print(validation[['includes_unsupported','includes_supported']].value_counts())

## 3. State the guarantee and limitations precisely

In [ ]:
print('This uses group-maximum binary-label nonconformity with exchangeable independent groups and a fixed predictor/candidate rule.')
print('A supported-only prediction set does not certify the error rate conditional on answering.')
print('Do not retune the candidate generator or reuse a tuned calibration score while claiming an unchanged finite-sample guarantee.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
